# ELI5 — how to self-code the models

A readable teaching notebook for a NumPy classifier and a tiny PyTorch CNN. It is intentionally not a production benchmark and does not create or replace production checkpoints. For self-training, use notebooks 01–03 or `12_model_family_benchmark.py`; those paths preserve the frozen raw split, validation-only selection, session manifest, and benchmark logs.


## 0. Setup

Run Jupyter from this notebook's folder, select the project kernel, and run the next cell. Put the dataset ZIP under data/raw.


In [ ]:
import importlib.util
import shutil
import subprocess
import sys

REQUIREMENTS = [
    "numpy>=1.26,<3",
    "pandas>=2.2,<3",
    "matplotlib>=3.9,<4",
    "seaborn>=0.13,<1",
    "pillow>=10,<12",
    "scikit-image>=0.24,<1",
    "scikit-learn>=1.5,<2",
    "torch>=2.4,<3",
    "torchvision>=0.19,<1",
    "tqdm>=4.66,<5",
    "joblib>=1.4,<2",
    "kaggle>=1.6,<2",
    "rich>=13.9,<15",
    "ipykernel>=6.29,<7",
    "jupyterlab>=4,<5",
    "notebook>=7.2,<8",
]
IMPORTS = {
    "numpy": "numpy", "pandas": "pandas", "matplotlib": "matplotlib",
    "seaborn": "seaborn", "pillow": "PIL", "scikit-image": "skimage",
    "scikit-learn": "sklearn", "torch": "torch", "torchvision": "torchvision",
    "joblib": "joblib", "kaggle": "kaggle", "rich": "rich",
}
missing = [package for package, module in IMPORTS.items() if importlib.util.find_spec(module) is None]
if missing:
    command = (
        ["uv", "pip", "install", "--python", sys.executable, *REQUIREMENTS]
        if shutil.which("uv")
        else [sys.executable, "-m", "pip", "install", *REQUIREMENTS]
    )
    print("Missing:", ", ".join(missing))
    print("Running:", " ".join(command))
    subprocess.check_call(command)
else:
    print("Core dependencies are already available in this kernel.")


## 1. Helpers and configuration


In [ ]:
from __future__ import annotations

import os
import random
import warnings
import zipfile
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from PIL import Image
from rich.console import Console
from rich.panel import Panel
from sklearn.model_selection import train_test_split
from torch import nn
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms

console = Console()
warnings.filterwarnings("ignore", message="CUDA initialization: Unexpected error")
RUN_PROFILE = os.getenv("COMPCLASS_PROFILE", "smoke")
SEED = int(os.getenv("COMPCLASS_SEED", "42"))
WORK_DIR = Path(os.getenv("COMPCLASS_WORK_DIR", Path.cwd())).expanduser().resolve()
RAW_DIR = WORK_DIR / "data" / "raw"
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff"}
IMAGE_SIZE = 64

def is_image(path):
    return path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS

def find_root(root):
    candidates = []
    for directory in [root, *sorted(path for path in root.rglob("*") if path.is_dir())]:
        children = [path for path in directory.iterdir() if path.is_dir() and not path.name.startswith(".")]
        if len(children) >= 2 and sum(any(is_image(item) for item in child.rglob("*")) for child in children) >= 2:
            candidates.append(directory)
    return min(candidates, key=lambda path: len(path.parts)) if candidates else None

def resolve_root():
    RAW_DIR.mkdir(parents=True, exist_ok=True)
    root = find_root(RAW_DIR)
    if root:
        return root
    for zip_path in RAW_DIR.glob("*.zip"):
        with zipfile.ZipFile(zip_path) as archive:
            archive.extractall(RAW_DIR / zip_path.stem)
    root = find_root(RAW_DIR)
    if root:
        return root
    raise RuntimeError(f"No dataset found under {RAW_DIR}")

def open_rgb(path):
    with Image.open(path) as image:
        if image.mode == "P" and "transparency" in image.info:
            image = image.convert("RGBA")
        image = image.convert("RGB")
        image.load()
        return image.copy()

def list_rows(root):
    return [(path, child.name) for child in sorted(root.iterdir()) if child.is_dir() for path in sorted(child.rglob("*")) if is_image(path)]

class ImageDataset(Dataset):
    def __init__(self, rows, labels):
        self.rows = rows
        self.to_index = {label: index for index, label in enumerate(labels)}
        self.transform = transforms.Compose([transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)), transforms.ToTensor()])
    def __len__(self):
        return len(self.rows)
    def __getitem__(self, index):
        path, label = self.rows[index]
        return self.transform(open_rgb(path)), self.to_index[label]

class TinyCNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(3, 8, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(8, 16, 3, padding=1), nn.ReLU(), nn.AdaptiveAvgPool2d(1),
            nn.Flatten(), nn.Linear(16, num_classes),
        )
    def forward(self, inputs):
        return self.net(inputs)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
console.print(Panel.fit(f"profile={RUN_PROFILE} data={RAW_DIR}", title="ELI5 model lab"))


## 2. The learning loop

A model makes a prediction, a loss measures how wrong it is, gradients show which parameter changes would reduce that loss, and an optimizer nudges the parameters. Repeating that loop is training.


In [ ]:
rng = np.random.default_rng(SEED)
x = rng.normal(size=(120, 2)).astype(np.float32)
y = (x[:, 0] + x[:, 1] > 0).astype(np.int64)
weights = np.zeros(2, dtype=np.float32)
bias = 0.0
losses = []
for _ in range(100):
    logits = x @ weights + bias
    probabilities = 1 / (1 + np.exp(-logits))
    error = probabilities - y
    losses.append(float(np.mean(-(y * np.log(probabilities + 1e-7) + (1 - y) * np.log(1 - probabilities + 1e-7)))))
    weights -= 0.2 * (x.T @ error) / len(x)
    bias -= 0.2 * float(np.mean(error))
console.print(f"weights={weights.round(2)} bias={bias:.2f} final loss={losses[-1]:.3f}")
plt.plot(losses)
plt.title("NumPy loss curve")
plt.xlabel("step")
plt.ylabel("loss")
plt.show()

root = resolve_root()
rows = list_rows(root)
wanted = [label for label in ("capacitor", "resistor", "transistor") if any(row_label == label for _, row_label in rows)]
if len(wanted) < 2:
    wanted = sorted({label for _, label in rows})[:3]
rows = [(path, label) for path, label in rows if label in wanted]
train_rows, test_rows = train_test_split(rows, test_size=0.25, random_state=SEED, stratify=[label for _, label in rows])
if RUN_PROFILE == "smoke":
    train_rows = sum(([row for row in train_rows if row[1] == label][:3] for label in wanted), [])
    test_rows = sum(([row for row in test_rows if row[1] == label][:1] for label in wanted), [])
labels = sorted({label for _, label in train_rows})
loaders = {
    "train": DataLoader(ImageDataset(train_rows, labels), batch_size=4, shuffle=True),
    "test": DataLoader(ImageDataset(test_rows, labels), batch_size=4),
}
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = TinyCNN(len(labels)).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_function = nn.CrossEntropyLoss()
for epoch in range(1 if RUN_PROFILE == "smoke" else 5):
    model.train()
    total_loss = 0.0
    for images, targets in loaders["train"]:
        images, targets = images.to(device), targets.to(device)
        optimizer.zero_grad()
        loss = loss_function(model(images), targets)
        loss.backward()
        optimizer.step()
        total_loss += float(loss.item())
    console.print(f"epoch={epoch + 1} loss={total_loss / max(len(loaders['train']), 1):.3f}")
model.eval()
images, targets = next(iter(loaders["test"]))
with torch.no_grad():
    probabilities = model(images.to(device)).softmax(1).cpu()
predictions = probabilities.argmax(1)
console.print(f"labels={labels}")
console.print(f"true indexes={targets.tolist()}")
console.print(f"predicted indexes={predictions.tolist()}")
console.print(f"top scores={probabilities.max(1).values.tolist()}")


This notebook is educational only. Use `$teach` for an interactive lesson on one cell. For reproducible training, run from `notebooks/` and start with the session manifest produced by notebooks 01–03 or the resumable family benchmark.
